# Sentiment Analysis of Cyberpunk 2077 — Steam Reviews### A Natural Language Processing Project**Project Goal:** Apply 15 NLP methods from our lecture to analyse ~92,000 Steam reviews of Cyberpunk 2077 and answer:  *"How did player sentiment evolve, and what specific aspects drove the game's dramatic shift from a troubled launch to one of Steam's highest-rated titles?"***Methods used (all from the NLP lecture):**| # | Method | Library ||---|--------|---------|| 1 | Tokenization | spaCy || 2 | Stop-word removal | spaCy || 3 | Lemmatization | spaCy || 4 | Stemming (Porter) | NLTK Snowball (comparison) || 5 | POS Tagging | spaCy || 6 | Named Entity Recognition | spaCy || 7 | Pattern Matching | spaCy Matcher || 8 | N-gram Analysis | sklearn CountVectorizer || 9 | tf-idf Vectorization | sklearn TfidfVectorizer || 10 | Levenshtein Distance | python-Levenshtein || 11 | N-gram Similarity (Dice/Jaccard) | manual implementation || 12 | Sentiment Classification | sklearn LogisticRegression || 13 | Word Embeddings (Word2Vec) | gensim || 14 | Search Engine (tf-idf cosine) | sklearn || 15 | RAG (Retrieval-Augmented Generation) | Search Engine + LLM API |

---## Phase 0 — Data LoadingLoad the dataset of ~92,000 English-language Steam reviews scraped via the Steam API.  Each review has: the review text, a thumbs-up/down vote, helpfulness votes, playtime, and a timestamp.

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Load the scraped Steam reviews
df = pd.read_csv('steam_reviews_dataset.csv')

# Quick overview
print(f"Dataset shape: {df.shape}")
print(f"\nPositive reviews: {df['voted_up'].sum():,}")
print(f"Negative reviews: {(~df['voted_up']).sum():,}")
print(f"Null reviews:     {df['review_text'].isna().sum()}")

# Drop rows with empty review text
df = df.dropna(subset=['review_text'])
print(f"\nAfter dropping nulls: {len(df):,} reviews")

# Add a readable date column from the Unix timestamp
df['date'] = pd.to_datetime(df['timestamp_created'], unit='s')
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

df.head()

---## Phase 1 — Text Preprocessing with spaCyAll preprocessing uses **spaCy**, the library taught in our lecture.  The spaCy pipeline runs tokenization, lemmatization, POS tagging, and NER in a single pass — this is more efficient than calling separate NLTK functions.### Method 1: Tokenization**Definition:** Tokenization is the segmentation of text into units at the word level.  We use `nlp()` which runs the full spaCy pipeline and gives us Token objects.### Method 2: Stop-word removal  **Definition:** Stop words are frequent words (the, is, and) that carry no content meaning.  spaCy marks them with `token.is_stop`.### Method 3: Lemmatization  **Definition:** Lemmatization finds the dictionary base form of a word (e.g., "running" → "run").  spaCy provides this via `token.lemma_`.

In [ ]:
import spacy

# Load the English spaCy model (small model — fast, good enough for our task)
# If not installed: python -m spacy download en_core_web_sm
nlp = spacy.load('en_core_web_sm')

# ============================================================
# DEMONSTRATION: Show what spaCy gives us for a single review
# ============================================================
sample = "The graphics are amazingly beautiful but the game constantly crashes on my PS5."
doc = nlp(sample)

print("=" * 80)
print("DEMONSTRATION: What spaCy extracts from a single sentence")
print("=" * 80)
print(f"\nOriginal: '{sample}'\n")
print(f"{'Token':<15} {'Lemma':<15} {'POS':<8} {'Stop?':<7} {'Entity'}")
print("-" * 65)
for token in doc:
    print(f"{token.text:<15} {token.lemma_:<15} {token.pos_:<8} {str(token.is_stop):<7} {token.ent_type_ if token.ent_type_ else '-'}")

print(f"\nEntities found: {[(ent.text, ent.label_) for ent in doc.ents]}")

### Building the preprocessing pipelineOur `clean_review()` function does everything in one pass through spaCy:1. **Tokenize** the text (Method 1)2. **Remove stop words** (Method 2) — but we keep "not" because it flips sentiment!3. **Lemmatize** each remaining word (Method 3)4. Remove URLs, punctuation, numbers, and single-character tokens

In [ ]:
def clean_review(text):
    """
    Full preprocessing pipeline using spaCy.
    Returns cleaned text with lemmatized, non-stop words.
    """
    # Step 0: Basic cleanup before spaCy
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)                 # Keep only letters
    text = re.sub(r'\s+', ' ', text).strip()                  # Collapse whitespace
    
    # Step 1: Run spaCy pipeline (tokenization + lemmatization + POS + NER)
    doc = nlp(text)
    
    # Step 2-3: Filter stop words (keep "not"!) and lemmatize
    tokens = []
    for token in doc:
        if token.is_stop and token.text != 'not':
            continue                    # Skip stop words (except "not")
        if token.is_punct or token.is_space:
            continue                    # Skip punctuation and whitespace
        if len(token.lemma_) <= 1:
            continue                    # Skip single characters
        tokens.append(token.lemma_)     # Use the LEMMA (base form), not the raw word
    
    return ' '.join(tokens)

# Test it on our sample
sample = "The graphics are amazingly beautiful but the game constantly crashes on my PS5."
cleaned = clean_review(sample)
print(f"Original:  {sample}")
print(f"Cleaned:   {cleaned}")
print(f"\nWhat happened:")
print(f"  'graphics'     → kept (content word)")
print(f"  'are'          → removed (stop word)")
print(f"  'amazingly'    → 'amazingly' (lemma)")
print(f"  'beautiful'    → 'beautiful' (lemma)")
print(f"  'but'          → removed (stop word)")  
print(f"  'crashes'      → 'crash' (lemmatized!)")
print(f"  'PS5'          → removed (has digits)")

### Apply preprocessing to all 92,000 reviewsWe use `nlp.pipe()` for batch processing — it processes reviews in parallel, which is much faster than calling `nlp()` on each review individually.  We also disable the NER and parser components during cleaning to speed things up.

In [ ]:
# For the cleaning step, we disable components we don't need (faster)
# We'll run the full pipeline later when we need NER and POS
print("Cleaning all reviews with spaCy... (this takes a few minutes)")

# Process in batches for speed — disable what we don't need for cleaning
nlp_fast = spacy.load('en_core_web_sm', disable=['ner', 'parser'])

def clean_review_fast(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    doc = nlp_fast(text)
    tokens = [token.lemma_ for token in doc 
              if not (token.is_stop and token.text != 'not')
              and not token.is_punct and not token.is_space
              and len(token.lemma_) > 1]
    return ' '.join(tokens)

df['clean_text'] = df['review_text'].apply(clean_review_fast)

# Drop any reviews that became empty after cleaning
df = df[df['clean_text'].str.strip().astype(bool)]

# Show before/after examples
print(f"\nDone! {len(df):,} reviews cleaned.\n")
print("=" * 80)
print("BEFORE vs AFTER examples:")
print("=" * 80)
for i in range(3):
    print(f"\nOriginal: {df['review_text'].iloc[i][:120]}...")
    print(f"Cleaned:  {df['clean_text'].iloc[i][:120]}...")

---### Method 4: Stemming (Porter Stemmer) — Comparison with Lemmatization**Definition from the lecture:**- **Stemming** = chops off word endings using rules. The result may NOT be a real word.- **Lemmatization** = looks up the dictionary base form. The result is ALWAYS a real word.We show both side-by-side on real review words to demonstrate the difference.This is a key distinction Prof. Bantel teaches — showing we understand it.

In [ ]:
from nltk.stem import SnowballStemmer

# Porter Stemmer (the one from the lecture — Snowball framework)
stemmer = SnowballStemmer('english')

# Pick words that show the difference clearly
test_words = ['crashes', 'crashing', 'crashed',          # verb forms
              'beautiful', 'beautifully',                  # adjective/adverb
              'running', 'ran', 'runs',                    # irregular verb
              'better', 'best',                            # comparative
              'studies', 'studying', 'studied',            # -ies/-ying
              'graphics', 'graphically',                   # noun/adverb
              'bugs', 'bugged', 'buggy']                   # game-relevant!

print("=" * 70)
print("STEMMING vs LEMMATIZATION — Side-by-side comparison")
print("=" * 70)
print(f"\n{'Word':<15} {'Stem (Porter)':<18} {'Lemma (spaCy)':<18} {'Same?'}")
print("-" * 60)

differences = 0
for word in test_words:
    stem = stemmer.stem(word)
    lemma = nlp(word)[0].lemma_
    same = "✓" if stem == lemma else "✗ DIFFERENT"
    if stem != lemma:
        differences += 1
    print(f"{word:<15} {stem:<18} {lemma:<18} {same}")

print(f"\n→ {differences} out of {len(test_words)} words differ between stemming and lemmatization.")
print("\nKey takeaway:")
print("  Stemming is FASTER but crude — 'studies' → 'studi' (not a real word!)")
print("  Lemmatization is SMARTER — 'studies' → 'study' (a real dictionary word)")
print("  For sentiment analysis, lemmatization is better because it preserves meaning.")

---### Method 5: Part-of-Speech (POS) Tagging**Definition:** POS tagging assigns each word its grammatical function (noun, verb, adjective...).  This is done automatically by spaCy's pipeline using `token.pos_`.We use POS tagging to answer: *"What adjectives do players use most in positive vs negative reviews?"*This connects POS tagging directly to sentiment analysis.

In [ ]:
from collections import Counter

print("=" * 70)
print("POS TAGGING — What adjectives define positive vs negative reviews?")
print("=" * 70)

# Separate positive and negative reviews
pos_reviews = df[df['voted_up'] == True]['review_text'].head(5000).tolist()
neg_reviews = df[df['voted_up'] == False]['review_text'].head(5000).tolist()

def extract_adjectives(texts, label):
    """Extract all adjectives from a list of texts using spaCy POS tagging."""
    adjs = []
    for doc in nlp.pipe(texts, batch_size=200, disable=['ner']):
        for token in doc:
            if token.pos_ == 'ADJ' and not token.is_stop and len(token.text) > 2:
                adjs.append(token.lemma_.lower())
    return Counter(adjs)

print("\nExtracting adjectives from positive reviews...")
pos_adjs = extract_adjectives(pos_reviews, "positive")
print("Extracting adjectives from negative reviews...")
neg_adjs = extract_adjectives(neg_reviews, "negative")

print(f"\n{'Rank':<6} {'Positive reviews (ADJ)':<30} {'Negative reviews (ADJ)'}")
print("-" * 65)
pos_top = pos_adjs.most_common(15)
neg_top = neg_adjs.most_common(15)
for i in range(15):
    p_word, p_count = pos_top[i] if i < len(pos_top) else ('', 0)
    n_word, n_count = neg_top[i] if i < len(neg_top) else ('', 0)
    print(f"  {i+1:<4} {p_word:<20} ({p_count:>4}x)    {n_word:<20} ({n_count:>4}x)")

print("\n→ Positive reviews use: 'great', 'good', 'beautiful', 'amazing'...")
print("→ Negative reviews use: 'bad', 'boring', 'broken', 'unplayable'...")
print("→ POS tagging lets us isolate opinion-carrying words (adjectives) from noise.")

### Demonstration: The full spaCy pipeline on a single reviewLet's show what spaCy extracts from one real review — tokenization, lemmatization, POS tags, and named entities, all in one pass.

In [ ]:
# Pick a rich review for demonstration
demo_text = "CD Projekt Red really fixed Cyberpunk 2077 after the disastrous PlayStation launch in December 2020."
doc = nlp(demo_text)

print("=" * 80)
print("FULL spaCy PIPELINE — Single review demonstration")
print("=" * 80)
print(f"\nOriginal text: '{demo_text}'\n")

# Tokenization + Lemmatization + POS
print(f"{'#':<4} {'Token':<14} {'Lemma':<14} {'POS':<6} {'Fine POS':<8} {'Stop?':<6} {'Entity'}")
print("-" * 72)
for i, token in enumerate(doc):
    ent = token.ent_type_ if token.ent_type_ else '-'
    print(f"{i:<4} {token.text:<14} {token.lemma_:<14} {token.pos_:<6} {token.tag_:<8} {str(token.is_stop):<6} {ent}")

# Named Entity Recognition (Method 6)
print(f"\nNamed Entities found:")
for ent in doc.ents:
    print(f"  '{ent.text}' → {ent.label_} ({spacy.explain(ent.label_)})")

print(f"\n→ spaCy identified the company (CD Projekt Red = ORG),")
print(f"   the product (Cyberpunk 2077), the platform (PlayStation),")
print(f"   and the time reference (December 2020 = DATE).")
print(f"\n→ This is Methods 1, 3, 5, and 6 — all from ONE nlp() call!")

---### Method 6: Named Entity Recognition (NER) on the full dataset**Definition:** NER finds and classifies named entities (persons, organizations, places, dates) in text.  We scan thousands of reviews to discover what real-world entities players mention most.

In [ ]:
print("=" * 70)
print("NAMED ENTITY RECOGNITION — What do players mention?")
print("=" * 70)
print("\nScanning 10,000 reviews for named entities...\n")

entity_counts = Counter()
entity_examples = {}

reviews_for_ner = df['review_text'].dropna().head(10000).tolist()

for doc in nlp.pipe(reviews_for_ner, batch_size=200):
    for ent in doc.ents:
        if len(ent.text) > 1:
            key = (ent.text.strip(), ent.label_)
            entity_counts[key] += 1
            if key not in entity_examples:
                entity_examples[key] = ent.label_

# Show top entities by type
print(f"{'Entity':<30} {'Type':<12} {'Count':<8} {'What it means'}")
print("-" * 75)
for (text, label), count in entity_counts.most_common(20):
    explanation = spacy.explain(label)
    print(f"{text:<30} {label:<12} {count:<8} {explanation}")

print(f"\n→ Total unique entities found: {len(entity_counts):,}")
print("→ Players mention companies (CD Projekt Red), platforms (PS4/PS5/PC),")
print("   and time references — useful for tracking sentiment over time.")